# TD3 — Prétraitement EEG : des signaux connus aux signaux du dataset CL-Drive

## Objectifs pédagogiques

Ce TD est consacré au **prétraitement des signaux EEG**. Il prépare directement le projet final sur l'estimation de la charge cognitive du conducteur à partir du dataset CL-Drive.

À la fin du TD, vous devez être capables de :

1. expliquer pourquoi un signal EEG doit être prétraité ;
2. comprendre le rôle d'un filtre passe-bande ;
3. comprendre le rôle d'un filtre notch ;
4. tester une chaîne de prétraitement sur des signaux synthétiques connus ;
5. appliquer la même logique à des signaux EEG du dataset ;
6. comparer les signaux avant et après traitement dans le domaine temporel et fréquentiel ;
7. formaliser une fonction de prétraitement EEG réutilisable dans le projet.

Dans le papier CL-Drive, l'EEG est acquis avec un casque Muse à 4 canaux, échantillonné à **256 Hz**. Le prétraitement EEG indiqué par les auteurs utilise un filtre passe-bande Butterworth d'ordre 2 entre **0.4 Hz et 75 Hz**, puis un filtre notch à **60 Hz** avec facteur de qualité **30**.

## 1. Pourquoi prétraiter un signal EEG ?

Un signal EEG brut contient l'activité électrique mesurée au niveau du cuir chevelu, mais aussi des perturbations :

- dérive lente de la ligne de base ;
- bruit secteur 50 Hz ou 60 Hz ;
- bruit haute fréquence ;
- artefacts.

Le prétraitement ne sert pas à embellir le signal. Il sert à produire un signal plus exploitable pour la segmentation, l'extraction de features et la classification.

Chaîne visée :

$$
\text{EEG brut} \rightarrow \text{gestion des valeurs manquantes} \rightarrow \text{passe-bande} \rightarrow \text{notch} \rightarrow \text{EEG prétraité}
$$

### Question 1

Citer au moins trois sources d'artefacts dans un signal EEG brut.

### Réponse 

Les mouvements musculaires (clignement des yeux, mastication, tension du cou) qui génèrent des EMG de forte amplitude. La dérive lente de la ligne de base souvent due à la transpiration ou au contact électrode/peau qui évolue. Le bruit secteur à 50 ou 60 Hz qui s'infiltre par couplage électromagnétique avec le réseau électrique.


## 2. Tester le prétraitement sur des signaux connus

Avant d'appliquer une chaîne de traitement à un EEG réel, il est utile de la tester sur un signal synthétique dont on connaît exactement le contenu fréquentiel.

On construira un signal contenant :

$$
x(t) = x_{lent}(t) + x_{alpha}(t) + x_{beta}(t) + x_{secteur}(t) + b(t)
$$

avec :

- une dérive lente à 0.2 Hz ;
- une composante alpha à 10 Hz ;
- une composante beta à 25 Hz ;
- une composante de bruit secteur à 50 Hz ou 60 Hz ;
- un bruit aléatoire.

Si le traitement est correct, la dérive lente et le bruit secteur doivent être fortement réduits, tandis que les composantes utiles doivent rester visibles.

### Question 2

Pourquoi commencer par un signal synthétique avant un signal EEG réel ?

### Réponse 

Parce qu'on connaît exactement ce qu'il contient. Si on applique un filtre passe bande censé supprimer le 0.2 Hz et le 60 Hz, on peut vérifier directement sur la PSD que ces composantes ont bien disparu et que les autres (10 Hz, 25 Hz) sont toujours là. Sur un EEG réel, on ne sait pas a priori ce qui doit rester et ce qui doit partir donc on ne peut pas valider aussi simplement que le filtre fait ce qu'on attend.


### Activité 1 — Générer un signal synthétique

Consignes méthodologiques :

1. créer un axe temporel de 20 secondes avec `fs = 256 Hz` ;
2. ajouter une sinusoïde lente à 0.2 Hz ;
3. ajouter une sinusoïde à 10 Hz ;
4. ajouter une sinusoïde à 25 Hz ;
5. ajouter une sinusoïde à 60 Hz ;
6. ajouter un bruit gaussien ;
7. tracer le signal temporel ;
8. calculer la PSD avec `scipy.signal.welch`.

Fonctions utiles : `np.arange`, `np.sin`, `np.random.normal`, `plt.plot`, `welch`.

In [ ]:
# Activité 1 — Implémenter ici la génération du signal synthétique.
# Suivre les étapes décrites dans la cellule précédente.

## 3. Filtre passe-bande Butterworth

Un filtre passe-bande conserve les fréquences situées dans une bande choisie et atténue les autres.

Dans CL-Drive, la bande EEG retenue est :

$$
0.4 \leq f \leq 75 \quad \text{Hz}
$$

Pour un signal échantillonné à $ f_s $, la fréquence de Nyquist vaut :

$$
f_N = \frac{f_s}{2}
$$

Les fréquences de coupure doivent être normalisées :

$$
W_{low}=\frac{f_{low}}{f_N}, \qquad W_{high}=\frac{f_{high}}{f_N}
$$

Fonctions utiles :

- `scipy.signal.butter` pour concevoir le filtre ;
- `scipy.signal.sosfiltfilt` pour filtrer sans déphasage notable ;
- `scipy.signal.welch` pour vérifier l'effet fréquentiel.

### Question 3

Pourquoi utiliser `filtfilt` ou `sosfiltfilt` pour une analyse hors ligne ?

### Réponse 

Le signal filtré est aligné temporellement avec le signal original ce qui est essentiel en analyse hors ligne car un déphasage décalerait les événements du signal par rapport aux labels.

### Question 4

Calculer la fréquence de Nyquist pour `fs = 256 Hz`.

### Réponse 

fN = 256 / 2 = 128 Hz

### Activité 2 — Algorithme du filtre passe-bande

Écrire une fonction de filtre passe-bande.

Algorithme :

1. fixer `lowcut = 0.4` ;
2. fixer `highcut = 75` ;
3. fixer `order = 2` ;
4. calculer `nyquist = fs / 2` ;
5. normaliser les fréquences ;
6. créer le filtre avec `butter(..., output="sos")` ;
7. appliquer le filtre avec `sosfiltfilt` ;
8. comparer signal brut et signal filtré.

In [ ]:
# Activité 2 — Implémenter ici le filtre passe-bande.

## 4. Filtre notch

Un filtre notch est contrôlé par :

- sa fréquence cible `notch_freq` ;
- son facteur de qualité `Q`.

Dans le papier, `Q = 30`.

Fonctions utiles : `scipy.signal.iirnotch`, puis `scipy.signal.filtfilt`.

### Question 5

Pourquoi CL-Drive utilise-t-il un notch à 60 Hz alors qu'en France on utilise plutôt 50 Hz ?

### Réponse 

Parce que le dataset CL Drive a été collecté au Canada (Queen's University, Kingston). Le réseau électrique nord-américain fonctionne à 60 Hz, contrairement au réseau européen qui est à 50 Hz. Le notch doit correspondre à la fréquence du réseau local au moment de l'acquisition.

### Question 6

Pourquoi ne faut-il pas appliquer automatiquement un notch sans regarder la PSD ?

### Réponse 

Parce qu'on risque de supprimer une composante utile si le bruit secteur n'est pas présent, ou si le signal contient une vraie activité cérébrale proche de 60 Hz (bande gamma haute). De plus, si le pic secteur est très large, le notch peut ne pas suffire. Donc regarder la PSD d'abord permet de confirmer la présence du pic et de choisir le bon paramètre Q.

### Activité 3 — Algorithme du notch

1. choisir `notch_freq` ;
2. choisir `Q = 30` ;
3. créer le filtre avec `iirnotch` ;
4. appliquer `filtfilt` ;
5. comparer les PSD avant/après ;
6. vérifier que le pic à 60 Hz diminue.

In [ ]:
# Activité 3 — Implémenter ici le filtre notch.

## 5. Fonction complète de prétraitement EEG

La fonction finale doit enchaîner :

1. conversion en tableau numérique ;
2. gestion des valeurs manquantes courtes ;
3. filtrage passe-bande ;
4. filtrage notch ;
5. retour du signal prétraité.

Paramètres recommandés pour CL-Drive :

| Paramètre | Valeur |
|---|---:|
| `fs` | 256 Hz |
| `lowcut` | 0.4 Hz |
| `highcut` | 75 Hz |
| `order of filter` | 2 |
| `notch_freq` | 60 Hz |
| `quality_factor` | 30 |

In [ ]:
# Activité 4 — Implémenter ici une fonction preprocess_eeg_1d.
# Elle doit gérer les NaN courts, appliquer le passe-bande, puis le notch.

### Question 7

Quel effet attend-on du passe-bande sur la dérive lente à 0.2 Hz ?

### Réponse 

On attend que la composante à 0.2 Hz (dérive lente) soit fortement atténuée et que les composantes à 10 Hz et 25 Hz soient préservées.

### Question 8

Quel effet attend-on du notch sur une composante à 60 Hz ?

### Réponse 

Le pic à 60 Hz dans la PSD doit être fortement réduit (quasiment éliminé), sans que les fréquences voisines soient trop affectées.

### Question 9

Pourquoi un highcut de 75 Hz est-il valide avec fs = 256 Hz ?

### Réponse 

75 < 128(fréquence de Nyquist) , donc c'est valide et on reste bien en dessous du repliement spectral.

### Question 10

Que se passe-t-il si `highcut > fs/2` ?

### Réponse 

Si on essayait de forcer la normalisation, la fréquence normalisée dépasserait 1, ce qui n'a pas de sens pour un filtre numérique. En pratique, scipy lève une exception avant même d'appliquer le filtre.

## 6. Passage au signal EEG multicanal

Dans CL-Drive, les canaux EEG principaux sont AF7, AF8, TP9 et TP10.

Le prétraitement doit être appliqué **canal par canal**.

Algorithme pour un DataFrame :

1. identifier la colonne temporelle si elle existe ;
2. identifier les colonnes EEG numériques ;
3. pour chaque canal : appliquer `preprocess_eeg_1d` ;
4. conserver la colonne temporelle ;
5. retourner un DataFrame filtré.

In [ ]:
# Activité 5 — Implémenter ici une fonction preprocess_eeg_dataframe.

## 7. Application aux signaux EEG du dataset CL-Drive

Après validation sur signaux synthétiques, appliquer la chaîne à un fichier EEG réel :

1. choisir un fichier EEG CSV ;
2. charger avec `pd.read_csv` ;
3. identifier les colonnes EEG ;
4. afficher un extrait du signal brut ;
5. appliquer le prétraitement ;
6. afficher le même extrait après traitement ;
7. comparer la PSD avant/après.

Si le dataset n'est pas encore disponible localement, le notebook doit rester générique et indiquer clairement où définir le chemin.

### Activité 6 — Charger et visualiser un fichier EEG réel

Fonctions utiles :

| Objectif | Fonction |
|---|---|
| Chemin de fichier | `pathlib.Path` |
| Lecture CSV | `pd.read_csv` |
| Affichage des colonnes | `df.columns` |
| Tracé temporel | `plt.plot` |
| PSD | `welch` |

In [ ]:
# Activité 6 — Charger ici un fichier EEG du dataset CL-Drive.

### Activité 7 — Comparaison avant/après sur EEG réel

Questions à traiter :

1. Le signal est-il plus exploitable après filtrage ?
2. Les variations très lentes sont-elles réduites ?
3. Observe-t-on une réduction autour de la fréquence secteur ?
4. Les canaux EEG ont-ils des comportements similaires ?
5. La PSD confirme-t-elle ce qui est visible dans le domaine temporel ?

In [ ]:
# Activité 7 — Visualiser ici les signaux EEG avant/après et leurs PSD.

### Question 11

Pourquoi faut-il comparer domaine temporel et domaine fréquentiel ?

### Réponse 

Les deux apportent des informations complémentaires. Le domaine temporel montre si la forme générale du signal est cohérente, si des artefacts ponctuels ont disparu, si l'amplitude est raisonnable. Le domaine fréquentiel (PSD) montre précisément si les bandes qu'on voulait supprimer ont bien disparu et si les bandes utiles sont toujours présentes.

### Question 12

Pourquoi ne faut-il pas interpoler une coupure EEG très longue ?

### Réponse 

Une coupure longue (plusieurs secondes) signifie que le signal est absent pendant un temps suffisant pour contenir plusieurs fenêtres entières. Interpoler crée des valeurs artificielles qui ne correspondent à aucune activité cérébrale réelle. Le modèle va apprendre sur des données fabriquées et produire des résultats biaisés donc il vaut mieux exclure ces segments.


### Question 13

Pourquoi le prétraitement doit-il être identique pour train et test ?

### Réponse 

C'est une forme classique de fuite de données ou de mismatch train/test. Si on applique un filtre différent en test, les distributions de features changent, et le modèle (entraîné sur d'autres distributions) se trompe systématiquement.

## 8. Contrôle qualité après prétraitement

Vérifications minimales :

1. absence de NaN ;
2. longueur inchangée ;
3. amplitude minimale et maximale réalistes ;
4. écart-type non nul ;
5. PSD cohérente ;
6. décision d'exclure les segments trop dégradés.

In [ ]:
# Activité 8 — Implémenter un rapport qualité simple pour les canaux EEG.

## 9. Prétraitement par lot

Dans le projet final, on appliquera la même fonction à plusieurs fichiers EEG.

Algorithme :

1. définir le dossier EEG ;
2. parcourir les fichiers CSV ;
3. ignorer les fichiers déjà filtrés ;
4. charger chaque fichier ;
5. appliquer `preprocess_eeg_dataframe` ;
6. sauvegarder une copie `filtered_...csv` ;
7. produire un diagnostic.

Pour info, la structure de l'ensemble de données est la suivante :

```text
CL-Drive
    |----EEG 
         |----participant_ID_1
                      |----eeg_data_level_1
                      |----eeg_baseline_level_1
                      .
                      .
                      .
                      |----eeg_data_level_9
                      |----eeg_baseline_level_9
         .
         .
         .
         |----participant_ID_21
    |----EDA
    |----ECG
    |----Gaze
    |----Labels
```

In [ ]:
# Activité 9 — Implémenter une fonction de prétraitement par lot.
# Exemple à adapter :
# saved_files = run_batch_eeg_preprocessing(DATASET_ROOT /EEG, fs=256, notch_freq=60)